<a href="https://colab.research.google.com/github/sofiapetruk/gs-01/blob/main/global_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json

In [ ]:
with open('telemetria_espacial.json', 'r', encoding='utf-8') as arquivo:
    dados = json.load(arquivo)

In [ ]:
# Quais dados representam o estado da missão

modulos = dados['modulos_criticos']

for nome, info in modulos.items():
    print(f"Módulo: {nome}")
    print(f"Status: {info['status']}")
    print(f"Descrição: {info['descricao']}")
    print(f"Última verificação: {info['ultima_verificacao']}")
    print("-----------------------------------------------")




Módulo: suporte_vida
Status: 1
Descrição: Operacional
Última verificação: 2031-03-15T08:00:00Z
-----------------------------------------------
Módulo: energia
Status: 1
Descrição: Operacional
Última verificação: 2031-03-15T08:00:00Z
-----------------------------------------------
Módulo: comunicacao
Status: 0
Descrição: FALHA — link primário offline
Última verificação: 2031-03-15T07:45:00Z
-----------------------------------------------
Módulo: habitat
Status: 1
Descrição: Operacional
Última verificação: 2031-03-15T08:00:00Z
-----------------------------------------------
Módulo: laboratorio
Status: 1
Descrição: Operacional
Última verificação: 2031-03-15T08:00:00Z
-----------------------------------------------
Módulo: armazenamento
Status: 0
Descrição: ALERTA — pressurização abaixo do nominal
Última verificação: 2031-03-15T07:50:00Z
-----------------------------------------------


In [ ]:
# Utilizar variáveis booleanas ou valores 0/1 para indicar o funcionamento de módulos críticos
## status = 1  →  módulo ATIVO/OPERACIONAL   (True em booleano)
## status = 0  →  módulo INATIVO/COM FALHA   (False em booleano)

for nome, status in modulos.items():
  print(f'MÓDULO: {nome}')
  print(f'Status: {status['status']}')
  print('-----------------------------')

MÓDULO: suporte_vida
Status: 1
-----------------------------
MÓDULO: energia
Status: 1
-----------------------------
MÓDULO: comunicacao
Status: 0
-----------------------------
MÓDULO: habitat
Status: 1
-----------------------------
MÓDULO: laboratorio
Status: 1
-----------------------------
MÓDULO: armazenamento
Status: 0
-----------------------------


In [ ]:
# Criar uma tabela simples de status indicando quais módulos estão normais, em alerta ou críticos

## Regra da tabela de classificação:
###   status = 1  E  nenhum alerta       → NORMAL
###   status = 1  E  alerta na descrição → ALERTA
###   status = 0   -> CRÍTICO

print(f"{'MÓDULO':<20} {'STATUS':<8} {'CLASSIFICAÇÃO':<15}")
print("-" * 50)

for nome, info in modulos.items():
    status = info["status"]
    descricao = info["descricao"].upper()

    if status == 0:
        classificacao = "CRÍTICO"
    elif "ALERTA" in descricao or "FALHA" in descricao:
        classificacao = "ALERTA"
    else:
        classificacao = "NORMAL"

    print(f"{nome:<20} {status:<8} {classificacao:<15}")


MÓDULO               STATUS   CLASSIFICAÇÃO  
--------------------------------------------------
suporte_vida         1        NORMAL         
energia              1        NORMAL         
comunicacao          0        CRÍTICO        
habitat              1        NORMAL         
laboratorio          1        NORMAL         
armazenamento        0        CRÍTICO        


# Incluir pelo menos uma regra de interpretação baseada em sistemas numéricos, estados binários ou faixas de segurança.

### Um sistema real de telemetria não diz apenas "ok" ou "falhou".
Ele compara um VALOR MEDIDO com um INTERVALO ACEITÁVEL (faixa de segurança).
Se o valor sair da faixa → gera alerta ou entra em estado crítico.

### Exemplo real de faixa:
 Temperatura interna segura: entre 18°C e 26°C
  - Se temp = 21.4°C → DENTRO da faixa → NORMAL
  - Se temp = 32.0°C → FORA da faixa   → ALERTA

### O estado binário (0 ou 1) é o RESULTADO dessa comparação:
  dentro_da_faixa = 1  (True)
  fora_da_faixa   = 0  (False)

# Os dados da missão devem ser organizados em estruturas apropriadas, mostrando aplicação prática dos conceitos estudados.

## Requisitos específicos:

### Utilizar listas para armazenar séries de dados como geração, consumo ou temperatura ao longo do tempo;
### Utilizar fila para organizar alertas pendentes por ordem de chegada ou prioridade;
### Utilizar pilha para registrar os últimos eventos críticos analisados;
### Utilizar dicionário ou tabela hash para acessar rapidamente dados de módulos pelo nome;
### Representar a hierarquia da missão, por exemplo: energia (solar, eólica ou baterias) e habitat (oxigênio, temperatura e comunicação);
### Utilizar matriz ou lista de listas para representar leituras por horário e variável.

In [ ]:
# Utilizar listas para armazenar séries de dados como geração, consumo ou temperatura ao longo do tempo;

leituras = dados["energia"]["leituras"]

geracao_lista = []
consumo_lista = []

for leitura in leituras:
    geracao_lista.append(leitura["geracao"])
    consumo_lista.append(leitura["consumo"])

print(geracao_lista)
print(consumo_lista)


[18.4, 17.9, 10.2, 19.7, 21.3, 22.1]
[15.1, 14.8, 16.3, 17.2, 20.9, 23.7]


In [ ]:
# Utilizar fila para organizar alertas pendentes por ordem de chegada ou prioridade;

from collections import deque

fila_alertas = deque()

for evento in dados["log_eventos"]:
    if evento["tipo"] in ["ALERTA", "FALHA_SENSOR", "FALHA_CRITICA"]:
        fila_alertas.append(evento)

print("Alertas pendentes:")
for alerta in fila_alertas:
    print(f"[{alerta['horario']}] {alerta['tipo']} - {alerta['modulo']}")

Alertas pendentes:
[2031-03-15T01:30:00Z] ALERTA - armazenamento
[2031-03-15T03:47:00Z] FALHA_SENSOR - laboratorio
[2031-03-15T07:45:00Z] FALHA_CRITICA - comunicacao
[2031-03-15T08:10:00Z] ALERTA - radiacao


In [ ]:
# Utilizar pilha para registrar os últimos eventos críticos analisados;

pilha = []

for evento in dados["log_eventos"]:
    if evento["tipo"] in ["FALHA_CRITICA", "FALHA_SENSOR"]:
        pilha.append(evento)

        if len(pilha) > 3:
            pilha.pop(0)

In [ ]:
# Utilizar dicionário ou tabela hash para acessar rapidamente dados de módulos pelo nome;

status_modulos = {}

for nome, info in dados["modulos_criticos"].items():
    status_modulos[nome] = info["status"]

print(status_modulos)

{'suporte_vida': 1, 'energia': 1, 'comunicacao': 0, 'habitat': 1, 'laboratorio': 1, 'armazenamento': 0}


In [ ]:
# Representar a hierarquia da missão, por exemplo: energia (solar, eólica ou baterias) e habitat (oxigênio, temperatura e comunicação);
def imprimir_arvore(no, nivel=0):
    for chave, filhos in no.items():
        print("   " * nivel + "└── " + chave)

        if isinstance(filhos, dict):
            imprimir_arvore(filhos, nivel + 1)

imprimir_arvore(dados)


└── missao
└── data_referencia
└── fuso_horario
└── modulos_criticos
   └── suporte_vida
      └── status
      └── descricao
      └── ultima_verificacao
   └── energia
      └── status
      └── descricao
      └── ultima_verificacao
   └── comunicacao
      └── status
      └── descricao
      └── ultima_verificacao
   └── habitat
      └── status
      └── descricao
      └── ultima_verificacao
   └── laboratorio
      └── status
      └── descricao
      └── ultima_verificacao
   └── armazenamento
      └── status
      └── descricao
      └── ultima_verificacao
└── energia
   └── unidade_geracao
   └── unidade_consumo
   └── unidade_reserva
   └── leituras
└── variaveis_ambientais
   └── temperatura_interna
      └── valor
      └── unidade
      └── status
      └── horario
   └── temperatura_externa
      └── valor
      └── unidade
      └── status
      └── horario
   └── nivel_radiacao
      └── valor
      └── unidade
      └── status
      └── limite_seguro
      └── horar

In [ ]:
# Utilizar matriz ou lista de listas para representar leituras por horário e variável.

matriz_energia = []

for leitura in dados["energia"]["leituras"]:
    linha = [
        leitura["horario"],
        leitura["geracao"],
        leitura["consumo"],
        leitura["reserva"]
    ]

    matriz_energia.append(linha)

for linha in matriz_energia:
    print(linha)

['2031-03-15T00:00:00Z', 18.4, 15.1, 312.0]
['2031-03-15T02:00:00Z', 17.9, 14.8, 318.2]
['2031-03-15T04:00:00Z', 10.2, 16.3, 306.5]
['2031-03-15T06:00:00Z', 19.7, 17.2, 310.8]
['2031-03-15T10:00:00Z', 21.3, 20.9, 308.4]
['2031-03-15T14:00:00Z', 22.1, 23.7, 296.1]


# O sistema deve implementar regras lógicas para tomar decisões básicas sobre o estado operacional da missão.

## Requisitos específicos:

### Construir regras com IF, ELIF e ELSE para classificar a situação da missão;
### Utilizar operadores lógicos AND, OR e NOT em pelo menos três regras distintas;
### Apresentar no README uma expressão booleana principal do diagnóstico;
### Explicar em linguagem simples por qual motivo cada regra gera determinada ação ou diagnóstico.

In [ ]:
# Construir regras com IF, ELIF e ELSE para classificar a situação da missão

modulos_operacionais = []

for nome, info in dados["modulos_criticos"].items():
    if info["status"] == 1:
        modulos_operacionais.append(nome)

criterio_atendido = False

for leitura in dados["energia"]["leituras"]:

    geracao = leitura["geracao"]
    consumo = leitura["consumo"]
    reserva = leitura["reserva"]

    if (
        20 <= geracao <= 25
        and consumo < 20
        and reserva > 310
    ):
        criterio_atendido = True
        break

if criterio_atendido:
    print("=== MÓDULOS OPERACIONAIS ===")

    for modulo in modulos_operacionais:
        print(modulo)
else:
  print("Nenhum módulo atende aos critérios da missão.")

Nenhum módulo atende aos critérios da missão.


In [ ]:
# Utilizar operadores lógicos AND, OR e NOT em pelo menos três regras distintas

# AND
for leitura in dados["energia"]["leituras"]:

    if (
        20 <= leitura["geracao"] <= 25
        and leitura["consumo"] < 20
        and leitura["reserva"] > 310
    ):
        print("Sistema operacional")

# OR
if (
    dados["variaveis_ambientais"]["qualidade_comunicacao"]["status"] == "FALHA"
    or
    dados["variaveis_ambientais"]["nivel_radiacao"]["status"] == "elevado"
):
    print("ALERTA DE MISSÃO")

# NOT
for nome, modulo in dados["modulos_criticos"].items():

    if not modulo["status"]:
        print(f"{nome} fora de operação")

ALERTA DE MISSÃO
comunicacao fora de operação
armazenamento fora de operação


# Explicar em linguagem simples por qual motivo cada regra gera determinada ação ou diagnóstico.

# Regras de Classificação da Missão

## Objetivo

Determinar quais módulos críticos da estação espacial podem ser considerados aptos para operação e exibição no relatório final da missão.

## Critérios de Elegibilidade

Um módulo crítico somente será exibido caso todas as condições abaixo sejam atendidas simultaneamente.

### Status do Módulo

O módulo deve estar operacional.

**Regra:**

```text
status = 1
```

### Geração de Energia

A geração de energia da estação deve estar dentro da faixa operacional esperada.

**Regra:**

```text
20 kW ≤ geração ≤ 25 kW
```

### Consumo de Energia

O consumo energético deve permanecer abaixo do limite estabelecido para operação segura.

**Regra:**

```text
consumo < 20 kW
```

### Reserva de Energia

A reserva energética deve permanecer acima do nível mínimo de segurança.

**Regra:**

```text
reserva > 310 kWh
```

## Regra Consolidada

Um módulo será considerado elegível quando:

```text
status = 1
E
20 ≤ geração ≤ 25
E
consumo < 20
E
reserva > 310
```

## Resultado Esperado

### Condições Atendidas

Quando todas as regras forem satisfeitas:

* O módulo será considerado apto para operação.
* O módulo será exibido no relatório final da missão.

### Condições Não Atendidas

Quando qualquer uma das regras não for satisfeita:

* O módulo será considerado não elegível.
* O módulo não será exibido no relatório final.

## Estrutura de Decisão

```text
SE status = 1
    E geração entre 20 e 25
    E consumo menor que 20
    E reserva maior que 310
ENTÃO
    Exibir módulo no relatório
SENÃO
    Não exibir módulo
```


# Explicar em linguagem simples por qual motivo cada regra gera determinada ação ou diagnóstico.

# Regras Lógicas da Missão

## Regra 1 - Elegibilidade de Operação (AND)

Um módulo será considerado apto para operação somente quando todas as condições forem atendidas simultaneamente.

**Operador utilizado:** AND

```text
status = 1
E
20 ≤ geração ≤ 25
E
consumo < 20
E
reserva > 310
```

Representação lógica:

```python
if status == 1 and 20 <= geracao <= 25 and consumo < 20 and reserva > 310:
    print("Módulo elegível")
```

---

## Regra 2 - Identificação de Situações de Risco (OR)

A missão deverá emitir alerta caso exista falha de comunicação ou nível de radiação acima do limite seguro.

**Operador utilizado:** OR

```text
Comunicação em falha
OU
Radiação acima do limite seguro
```

Representação lógica:

```python
if comunicacao_falha or radiacao_elevada:
    print("ALERTA DE MISSÃO")
```

---

## Regra 3 - Verificação de Módulos Não Operacionais (NOT)

Um módulo será considerado indisponível quando não estiver operacional.

**Operador utilizado:** NOT

```text
NÃO operacional
```

Representação lógica:

```python
if not status:
    print("Módulo indisponível")
```

ou

```python
if not (status == 1):
    print("Módulo indisponível")
```

---

## Resumo dos Operadores Utilizados

| Operador | Finalidade                                  | Exemplo                                                                         |
| -------- | ------------------------------------------- | ------------------------------------------------------------------------------- |
| AND      | Todas as condições devem ser verdadeiras    | Status operacional e geração adequada e consumo controlado e reserva suficiente |
| OR       | Pelo menos uma condição deve ser verdadeira | Falha de comunicação ou radiação elevada                                        |
| NOT      | Inverte o resultado da condição             | Módulo não operacional                                                          |


# O sistema deve gerar alertas automáticos em resposta a situações críticas ou anômalas detectadas nos dados.

## Requisitos específicos:

### Implementar a geração de alertas para situações críticas como falha de módulos essenciais, energia baixa ou comunicação comprometida;
### Classificar alertas em níveis de severidade (normal, alerta ou crítico);
### Exibir os alertas de forma clara, organizada e priorizando os mais críticos;
### Fornecer recomendações automáticas de ações em resposta a cada alerta.

In [ ]:
class MonitorMissao:

    def __init__(self, dados):
        self.dados = dados
        self.alertas = []

    def adicionar_alerta(self, severidade, modulo, descricao, recomendacao):
        self.alertas.append({
            "severidade": severidade,
            "modulo": modulo,
            "descricao": descricao,
            "recomendacao": recomendacao
        })

    def verificar_modulos_criticos(self):

        for nome, modulo in self.dados["modulos_criticos"].items():

            if modulo["status"] == 0:

                self.adicionar_alerta(
                    "CRITICO",
                    nome,
                    modulo["descricao"],
                    "Realizar diagnóstico imediato do módulo."
                )

    def verificar_energia(self):

        ultima_leitura = self.dados["energia"]["leituras"][-1]

        reserva = ultima_leitura["reserva"]
        consumo = ultima_leitura["consumo"]

        if reserva < 300:

            self.adicionar_alerta(
                "CRITICO",
                "energia",
                f"Reserva crítica: {reserva} kWh",
                "Reduzir consumo dos módulos não essenciais."
            )

        elif reserva <= 310:

            self.adicionar_alerta(
                "ALERTA",
                "energia",
                f"Reserva baixa: {reserva} kWh",
                "Monitorar consumo energético."
            )

        if consumo > 23:

            self.adicionar_alerta(
                "ALERTA",
                "energia",
                f"Consumo elevado: {consumo} kW",
                "Reavaliar atividades de alto consumo."
            )

    def verificar_comunicacao(self):

        comunicacao = self.dados["variaveis_ambientais"]["qualidade_comunicacao"]

        if comunicacao["status"] == "FALHA":

            self.adicionar_alerta(
                "CRITICO",
                "comunicacao",
                comunicacao["observacao"],
                "Ativar comunicação de backup."
            )

    def verificar_radiacao(self):

        radiacao = self.dados["variaveis_ambientais"]["nivel_radiacao"]

        if radiacao["valor"] > radiacao["limite_seguro"]:

            self.adicionar_alerta(
                "ALERTA",
                "radiacao",
                f"Radiação em {radiacao['valor']} mSv/h",
                "Suspender atividades externas (EVA)."
            )

    def ordenar_alertas(self):

        prioridade = {
            "CRITICO": 1,
            "ALERTA": 2,
            "NORMAL": 3
        }

        self.alertas.sort(
            key=lambda alerta: prioridade[alerta["severidade"]]
        )

    def exibir_alertas(self):

        self.ordenar_alertas()

        print("\n=== ALERTAS DA MISSÃO ===\n")

        for alerta in self.alertas:

            print(f"SEVERIDADE : {alerta['severidade']}")
            print(f"MÓDULO     : {alerta['modulo']}")
            print(f"DESCRIÇÃO  : {alerta['descricao']}")
            print(f"AÇÃO       : {alerta['recomendacao']}")
            print("-" * 50)

    def executar_monitoramento(self):

        self.verificar_modulos_criticos()
        self.verificar_energia()
        self.verificar_comunicacao()
        self.verificar_radiacao()

        self.exibir_alertas()

monitor = MonitorMissao(dados)
monitor.executar_monitoramento()


=== ALERTAS DA MISSÃO ===

SEVERIDADE : CRITICO
MÓDULO     : comunicacao
DESCRIÇÃO  : FALHA — link primário offline
AÇÃO       : Realizar diagnóstico imediato do módulo.
--------------------------------------------------
SEVERIDADE : CRITICO
MÓDULO     : armazenamento
DESCRIÇÃO  : ALERTA — pressurização abaixo do nominal
AÇÃO       : Realizar diagnóstico imediato do módulo.
--------------------------------------------------
SEVERIDADE : CRITICO
MÓDULO     : energia
DESCRIÇÃO  : Reserva crítica: 296.1 kWh
AÇÃO       : Reduzir consumo dos módulos não essenciais.
--------------------------------------------------
SEVERIDADE : CRITICO
MÓDULO     : comunicacao
DESCRIÇÃO  : Antena principal sem sinal — usando backup com 34% de eficiência
AÇÃO       : Ativar comunicação de backup.
--------------------------------------------------
SEVERIDADE : ALERTA
MÓDULO     : energia
DESCRIÇÃO  : Consumo elevado: 23.7 kW
AÇÃO       : Reavaliar atividades de alto consumo.
---------------------------------

# A equipe deve aplicar uma técnica simples de análise ou previsão de dados para estimar o comportamento de uma variável crítica da missão.

## Requisitos específicos:

### Aplicar uma técnica simples como regressão linear, média móvel ou extrapolação de tendência sem bibliotecas avançadas;
### A variável analisada pode ser energia disponível, consumo, geração renovável, temperatura ou qualidade de comunicação;
### Mostrar claramente os dados utilizados, a metodologia aplicada e o resultado previsto;
### A previsão deve influenciar pelo menos uma recomendação ou decisão do sistema.

In [ ]:
print("=== ANÁLISE E PREVISÃO DE ENERGIA ===")

reservas = []

for leitura in dados["energia"]["leituras"]:
    reservas.append(leitura["reserva"])

print("\nDados utilizados:")
for reserva in reservas:
    print(reserva)

ultimas_tres = reservas[-3:]

soma = 0

for valor in ultimas_tres:
    soma += valor

previsao = soma / len(ultimas_tres)

print("\nMetodologia utilizada:")
print("Média móvel simples dos últimos 3 registros.")

print(f"\nPrevisão da próxima reserva: {previsao:.2f} kWh")

if previsao < 300:
    print("\nSituação: CRÍTICA")
    print("Recomendação: reduzir imediatamente o consumo de energia.")

elif previsao <= 310:
    print("\nSituação: ALERTA")
    print("Recomendação: monitorar o consumo e priorizar sistemas essenciais.")

else:
    print("\nSituação: NORMAL")
    print("Recomendação: manter operação normal.")

=== ANÁLISE E PREVISÃO DE ENERGIA ===

Dados utilizados:
312.0
318.2
306.5
310.8
308.4
296.1

Metodologia utilizada:
Média móvel simples dos últimos 3 registros.

Previsão da próxima reserva: 305.10 kWh

Situação: ALERTA
Recomendação: monitorar o consumo e priorizar sistemas essenciais.
